# Modelado predictivo y prescriptivo
**Optimización del portafolio de variedades y la asignación de superficie para maximizar el margen por metro cuadrado: el caso de una florícola exportadora de rosas**

Bryan Steve Vargas Cabascango — Maestría en Inteligencia de Negocios y Ciencia de Datos

---

Este cuaderno implementa las dos etapas del modelo. Requiere haber ejecutado antes
`01_EDA_Portafolio_Rosas.ipynb`, que deja el panel depurado en `figuras/panel_analitico.pkl`.

```
pip install pandas numpy matplotlib scikit-learn scipy openpyxl
```

## 1. Configuración y carga del panel analítico

In [ ]:
import warnings, os, json
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize

ENTRADA = './figuras/panel_analitico.pkl'   # generado por el cuaderno de EDA
SALIDA = './modelo/'
os.makedirs(SALIDA, exist_ok=True)

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,
                     'axes.grid': True, 'grid.alpha': .3, 'figure.dpi': 110,
                     'axes.spines.top': False, 'axes.spines.right': False})
AZ, RO, VE, MO, GR = '#4C72B0', '#C44E52', '#55A868', '#8172B2', '#BBBBBB'

def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(f'{SALIDA}{nombre}.png', dpi=150, bbox_inches='tight')
    plt.show()

d = pd.read_pickle(ENTRADA).sort_values(['variety', 'id_month'])
print(f'{len(d):,} observaciones | {d.variety.nunique()} variedades | '
      f'{d.id_month.nunique()} meses')

## 2. Variables de decisión y variables del modelo

**Variable de decisión del problema:** el vector $w$ de participaciones de superficie
asignadas a cada variedad, con $\sum_i w_i = 1$. Es la única variable sobre la que la
gerencia puede actuar.

**Variable objetivo del modelo predictivo:** el margen mensual por metro cuadrado
$m_{v,t}$, construido en el cuaderno de EDA.

**Variables predictoras:** se construyen exclusivamente con información disponible
hasta $t-1$ para evitar fuga temporal.

In [ ]:
g = d.groupby('variety', group_keys=False)

# --- persistencia del desempeno de la variedad ---------------------------
for L in (1, 2, 3, 12):
    d[f'lag{L}'] = g.margin_m2.shift(L)
d['roll3']    = g.margin_m2.transform(lambda s: s.shift(1).rolling(3).mean())
d['roll12']   = g.margin_m2.transform(lambda s: s.shift(1).rolling(12).mean())
d['exp_mean'] = g.margin_m2.transform(lambda s: s.shift(1).expanding().mean())
d['exp_std']  = g.margin_m2.transform(lambda s: s.shift(1).expanding().std())

# --- estado operativo y comercial del mes anterior -----------------------
d['lag_stems'] = g.stems_m2.shift(1)
d['lag_nac']   = g.pct_nacional.shift(1)
d['lag_nocol'] = g.pct_no_colocada.shift(1)
d['lag_price'] = g.avg_price.shift(1)

# --- calendario ----------------------------------------------------------
d['mes']   = d.id_month.dt.month
d['trend'] = (d.id_month.dt.year - 2020) * 12 + d.mes

FEATS = ['lag1','lag2','lag3','lag12','roll3','roll12','exp_mean','exp_std',
         'lag_stems','lag_nac','lag_nocol','lag_price','mes','trend',
         'density','monthly_flowers_per_plant','area','n_plants',
         'is_valentin','is_mothers_day_season','is_all_saints_season',
         'is_christmas_season','is_covid_shock']

dm = d.dropna(subset=['lag12','roll12','exp_mean','margin_m2']).copy()
dm['exp_std'] = dm.exp_std.fillna(0)
print(f'Observaciones modelables: {len(dm):,}  ({len(FEATS)} variables predictoras)')

### Justificación de la selección de variables

| Bloque | Variables | Fundamento en la literatura |
|---|---|---|
| Persistencia | rezagos 1, 2, 3 y 12; medias móviles 3 y 12; media y desviación históricas | Barkley et al. (2010) caracterizan cada variedad por su media y varianza históricas de desempeño. La media histórica expansiva es además la **línea base** del modelo. |
| Calendario comercial | mes, tendencia, indicadores de San Valentín, Día de la Madre, Difuntos, Navidad y choque COVID | Marko et al. (2016, 2017) condicionan el rendimiento esperado a escenarios exógenos. El EDA mostró que la variación entre meses explica el 31,2 % de la varianza. |
| Escala y agronomía | superficie, número de plantas, densidad, flores esperadas por planta | Recogen la heterogeneidad agronómica entre variedades que Barkley et al. atribuyen a la interacción genotipo-ambiente. |
| Estado comercial previo | precio, tallos/m², proporción nacional y no colocada del mes anterior | La proporción no colocada resultó el predictor negativo de mayor magnitud en el EDA (r = −0,396). |

## 3. Etapa predictiva

El modelo no pronostica: **descompone**. Separa el componente anticipable del margen
del residuo que constituye la incertidumbre genuina.

$$\varepsilon_{v,t} = m_{v,t} - \hat{m}_{v,t}$$

Se evalúan tres algoritmos de ensamble frente a dos líneas base, con partición temporal.

In [ ]:
MODELOS = {
  # Configuracion de produccion. La busqueda de hiperparametros
  # (03_Optimizacion_Hiperparametros.ipynb) la situa a menos de un error
  # estandar de la mejor, por lo que se conserva (regla de un error estandar).
  'Random Forest': lambda: RandomForestRegressor(
      n_estimators=500, min_samples_leaf=5, max_features=.5,
      random_state=42, n_jobs=-1),
  'Extra Trees': lambda: ExtraTreesRegressor(
      n_estimators=500, min_samples_leaf=5, max_features=.5,
      random_state=42, n_jobs=-1),
  'Gradient Boosting': lambda: GradientBoostingRegressor(
      n_estimators=400, learning_rate=.05, max_depth=4,
      subsample=.8, random_state=42),
}

def metricas(y, p):
    return dict(RMSE=np.sqrt(mean_squared_error(y, p)),
                MAE=mean_absolute_error(y, p),
                R2=r2_score(y, p))

CORTE = '2025-01-01'
tr, te = dm[dm.id_month < CORTE], dm[dm.id_month >= CORTE]
print(f'Entrenamiento: {len(tr):,} obs (hasta {tr.id_month.max():%Y-%m})')
print(f'Prueba:        {len(te):,} obs ({te.id_month.nunique()} meses)')

tabla = {
  'Media histórica (línea base)': metricas(te.margin_m2, te.exp_mean),
  'Media global (línea base)': metricas(te.margin_m2, np.full(len(te), tr.margin_m2.mean())),
}
ajustados = {}
for nombre, f in MODELOS.items():
    m = f().fit(tr[FEATS], tr.margin_m2)
    tabla[nombre] = metricas(te.margin_m2, m.predict(te[FEATS]))
    ajustados[nombre] = m

T = pd.DataFrame(tabla).T
base = T.loc['Media histórica (línea base)', 'RMSE']
T['Mejora RMSE %'] = ((base - T.RMSE) / base * 100)
T.round(4).to_csv(SALIDA + 'comparacion_modelos.csv')

MEJOR = T.drop(index=[i for i in T.index if 'línea base' in i]).RMSE.idxmin()
print(f'\nMejor modelo: {MEJOR}\n')
T.round(4)

### Figura 1. Desempeño predictivo fuera de muestra

In [ ]:
t = T.sort_values('RMSE')
col = [GR if 'línea base' in i else (VE if i == MEJOR else AZ) for i in t.index]

fig, ax = plt.subplots(figsize=(8, 4))
barras = ax.barh(range(len(t)), t.RMSE, color=col)
ax.set_yticks(range(len(t)))
ax.set_yticklabels([i.replace(' (línea base)', '\n(línea base)') for i in t.index], fontsize=9)
ax.invert_yaxis()
for i, val in enumerate(t.RMSE):
    ax.text(val + .008, i, f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('Raíz del error cuadrático medio (USD por m² al mes)')
ax.set_xlim(0, t.RMSE.max() * 1.15)
ax.set_title('Desempeño predictivo fuera de muestra')
guardar(fig, 'm01_comparacion_modelos')

### Figura 2. Capacidad explicativa

In [ ]:
t2 = T.sort_values('R2')
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(len(t2)), t2.R2, color=[GR if 'línea base' in i else AZ for i in t2.index])
ax.set_yticks(range(len(t2)))
ax.set_yticklabels([i.replace(' (línea base)', '\n(línea base)') for i in t2.index], fontsize=9)
ax.axvline(0, color='k', lw=.9)
for i, val in enumerate(t2.R2):
    ax.text(val + .008, i, f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Coeficiente de determinación fuera de muestra')
ax.set_title('Capacidad explicativa de cada modelo')
guardar(fig, 'm02_r2_modelos')

### Validación de ventana expansiva

La partición simple entrega solo 20 meses de residuos, insuficientes para estimar una
matriz de covarianzas. Se reajusta el modelo año por año para obtener residuos fuera
de muestra sobre un horizonte mayor.

In [ ]:
residuos = []
for anio in range(2022, 2027):
    ent = dm[dm.id_month.dt.year < anio]
    pru = dm[dm.id_month.dt.year == anio]
    if len(pru) == 0:
        continue
    m = MODELOS[MEJOR]().fit(ent[FEATS], ent.margin_m2)
    pred = m.predict(pru[FEATS])
    residuos.append(pru[['variety','id_month','margin_m2']].assign(
        pred=pred, resid=pru.margin_m2.values - pred))

R = pd.concat(residuos)
R.to_pickle(SALIDA + 'residuos.pkl')

mm = metricas(R.margin_m2, R.pred)
print(f'Ventana expansiva 2022-2026: {R.id_month.nunique()} meses, {len(R):,} obs')
print(f'  RMSE = {mm["RMSE"]:.4f}   R² = {mm["R2"]:.4f}')
print(f'  Residuo medio = {R.resid.mean():.4f}  (sesgo)')

### Figura 3. Importancia de variables

In [ ]:
ETQ = {'roll3':'Media móvil 3 meses','lag3':'Rezago 3 meses','trend':'Tendencia',
  'lag_price':'Precio mes anterior','mes':'Mes calendario','lag12':'Rezago 12 meses',
  'roll12':'Media móvil 12 meses','exp_mean':'Media histórica','lag1':'Rezago 1 mes',
  'lag2':'Rezago 2 meses','exp_std':'Desv. histórica','lag_stems':'Tallos/m² mes anterior',
  'lag_nocol':'No colocada mes anterior','lag_nac':'Nacional mes anterior',
  'density':'Densidad','area':'Superficie','n_plants':'N.º de plantas',
  'monthly_flowers_per_plant':'Flores por planta','is_valentin':'San Valentín',
  'is_mothers_day_season':'Día de la Madre','is_all_saints_season':'Difuntos',
  'is_christmas_season':'Navidad','is_covid_shock':'Choque COVID'}

imp = pd.Series(ajustados[MEJOR].feature_importances_, index=FEATS).sort_values(ascending=False)
imp.to_csv(SALIDA + 'importancia.csv')

t = imp.head(14)[::-1]
fig, ax = plt.subplots(figsize=(8, 5.4))
ax.barh([ETQ.get(i, i) for i in t.index], t.values, color=AZ)
ax.set_xlabel('Importancia relativa (reducción de impureza)')
ax.set_title('Variables más influyentes en el modelo seleccionado')
guardar(fig, 'm03_importancia_variables')
imp.head(10).round(4)

### Figura 4. Predicho frente a observado

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.4))
ax.scatter(R.pred, R.margin_m2, s=7, alpha=.22, color=AZ, edgecolor='none')
lim = [R.margin_m2.min(), R.margin_m2.max()]
ax.plot(lim, lim, 'k--', lw=1.2, label='Predicción perfecta')
ax.set_xlabel('Margen predicho (USD por m² al mes)')
ax.set_ylabel('Margen observado (USD por m² al mes)')
ax.set_title(f'Predicho frente a observado (R² = {r2_score(R.margin_m2, R.pred):.3f})')
ax.legend()
guardar(fig, 'm04_predicho_vs_observado')

### Figura 5. Distribución de los residuos

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(R.resid, bins=60, color=AZ, edgecolor='white', lw=.4)
ax.axvline(0, color='k', lw=1.2)
ax.axvline(R.resid.mean(), color=RO, ls='--', lw=1.5, label=f'Media = {R.resid.mean():.3f}')
ax.set_xlabel('Residuo (USD por m² al mes)'); ax.set_ylabel('Observaciones')
ax.set_title('Distribución de los residuos del modelo predictivo')
ax.legend()
guardar(fig, 'm05_distribucion_residuos')

### Figura 6. Verificación del supuesto de independencia serial

La anualización de la matriz de riesgo multiplica por doce, lo que exige ausencia de
dependencia serial en los residuos. **Este supuesto se verifica, no se asume.**

In [ ]:
M = R[R.id_month >= '2022-01-01'].pivot_table(index='id_month', columns='variety', values='resid')
M = M.loc[:, M.notna().mean() >= .95].fillna(0.)
VAR = list(M.columns)
print(f'Matriz de residuos: {M.shape[0]} meses x {M.shape[1]} variedades  '
      f'(T/N = {M.shape[0]/M.shape[1]:.2f})')

rezagos = range(1, 13)
acs = [np.nanmedian([M[v_].autocorr(k) for v_ in VAR]) for k in rezagos]
ic = 1.96 / np.sqrt(len(M))

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(rezagos, acs, color=[RO if abs(a) > ic else AZ for a in acs], width=.6)
ax.axhline(0, color='k', lw=.9)
ax.axhline(ic, color=RO, ls=':', lw=1.2, label='Banda de significancia al 95 %')
ax.axhline(-ic, color=RO, ls=':', lw=1.2)
ax.set_xticks(list(rezagos))
ax.set_xlabel('Rezago (meses)'); ax.set_ylabel('Autocorrelación mediana')
ax.set_title('Autocorrelación de los residuos por variedad')
ax.legend(fontsize=9)
guardar(fig, 'm06_autocorrelacion_residuos')
print(f'Autocorrelación mediana en el rezago 1: {acs[0]:+.4f}  (banda ±{ic:.3f})')

## 4. Etapa prescriptiva

### 4.1 Matriz de riesgo: encogimiento de Ledoit-Wolf

La covarianza muestral de $N$ variedades requiere $N(N+1)/2$ parámetros. Con $T/N < 1$
es singular, y Michaud (1989) mostró que en esas condiciones el optimizador se comporta
como un **maximizador de errores de estimación**.

$$\hat{\Sigma} = \delta F + (1-\delta)S$$

In [ ]:
lw = LedoitWolf().fit(M.values)
S_lw = lw.covariance_
S_sam = np.cov(M.values, rowvar=False)

ev_s = np.linalg.eigvalsh(S_sam)[::-1]
ev_l = np.linalg.eigvalsh(S_lw)[::-1]

print(f'Intensidad de encogimiento δ = {lw.shrinkage_:.4f}\n')
print(f'{"":26} {"muestral":>14} {"encogida":>14}')
print(f'{"autovalor mínimo":26} {ev_s.min():>14.3e} {ev_l.min():>14.3e}')
print(f'{"número de condición":26} {ev_s.max()/max(abs(ev_s.min()),1e-18):>14.2e} '
      f'{ev_l.max()/ev_l.min():>14.2f}')

### Figura 7. Condicionamiento de la matriz de covarianzas

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.semilogy(range(1, len(ev_s)+1), np.clip(ev_s, 1e-18, None), 'o-', ms=3,
            color=RO, label='Covarianza muestral')
ax.semilogy(range(1, len(ev_l)+1), ev_l, 'o-', ms=3, color=VE,
            label=f'Encogida (δ = {lw.shrinkage_:.3f})')
ax.set_xlabel('Índice del autovalor'); ax.set_ylabel('Autovalor (escala logarítmica)')
ax.set_title('Condicionamiento de la matriz de covarianzas')
ax.legend()
guardar(fig, 'm07_autovalores')

### Figura 8. Efecto de la descomposición sobre la medida de riesgo

In [ ]:
H = d[d.id_month >= '2022-01-01'].pivot_table(
        index='id_month', columns='variety', values='margin_m2')[VAR]

fig, ax = plt.subplots(figsize=(6.2, 5.4))
ax.scatter(H.std().values, np.sqrt(np.diag(S_lw)), s=22, alpha=.7,
           color=AZ, edgecolor='k', lw=.3)
lim = [0, max(H.std().max(), np.sqrt(np.diag(S_lw)).max()) * 1.05]
ax.plot(lim, lim, 'k--', lw=1.2, label='Sin reducción')
ax.set_xlabel('Desv. estándar del margen crudo (USD por m² al mes)')
ax.set_ylabel('Desv. estándar del residuo (USD por m² al mes)')
ax.set_title('Efecto de la descomposición sobre la medida de riesgo')
ax.legend()
guardar(fig, 'm08_riesgo_residual_vs_crudo')
print(f'Reducción media de la desviación estándar: '
      f'{(1 - np.sqrt(np.diag(S_lw)).mean()/H.std().mean())*100:.1f} %')

### 4.2 Programa cuadrático

$$\min_w \; w'\hat{\Sigma}w \quad \text{s.a.}\quad
\mu'w \ge \theta,\;\;
\sum_i w_i = 1,\;\;
0 \le w_i \le u_i,\;\;
\tfrac{1}{2}\sum_i |w_i - w^0_i| \le \tau$$

con $u_i = \min(0{,}25;\; \max(2w^0_i;\;0{,}005))$ y $\tau = 0{,}20$.

La restricción de rotación es una desigualdad en norma $L_1$, no diferenciable, por lo
que se reformula con variables auxiliares $u_i \ge |w_i - w^0_i|$.

In [ ]:
CAP, CRECE, TURNOVER = 0.25, 2.0, 0.20

# --- retorno esperado: prediccion del modelo sobre los ultimos 12 meses ---
P = R[R.id_month >= '2022-01-01'].pivot_table(index='id_month', columns='variety', values='pred')[VAR]
mu_a = P.tail(12).mean().values * 12          # anualizado
S_a = S_lw * 12
mu_hist = H.mean().values * 12                # estimador alternativo (Barkley et al.)

# --- cartera vigente ------------------------------------------------------
ultimo = d.id_month.max()
area = d[d.id_month == ultimo].set_index('variety')['area'].reindex(VAR).fillna(0)
w0 = (area / area.sum()).values
AREA_TOTAL = float(area.sum())
print(f'Cartera vigente ({ultimo:%Y-%m}): {AREA_TOTAL:,.0f} m² en {(w0>1e-4).sum()} variedades')
print(f'Participación máxima: {w0.max()*100:.1f} %')

In [ ]:
def _cotas(w0):
    """Techo por variedad: no mas del doble de su superficie actual."""
    return np.minimum(CAP, np.maximum(w0 * CRECE, 0.005))

def _planteo(mu, w0, turnover):
    n = len(mu); ub = _cotas(w0)
    if turnover is None:
        return ([{'type':'eq','fun': lambda z: z.sum() - 1}],
                list(zip(np.zeros(n), ub)), (lambda z: z), w0.copy())
    cons = [{'type':'eq',  'fun': lambda z: z[:n].sum() - 1},
            {'type':'ineq','fun': lambda z: z[n:] - (z[:n] - w0)},
            {'type':'ineq','fun': lambda z: z[n:] + (z[:n] - w0)},
            {'type':'ineq','fun': lambda z: 2*turnover - z[n:].sum()}]
    return (cons, list(zip(np.zeros(n), ub)) + [(0,1)]*n,
            (lambda z: z[:n]), np.concatenate([w0, np.zeros(n)]))

def _resolver(obj, cons, bnds, z0, extra=()):
    for ftol in (1e-9, 1e-7, 1e-5):
        r = minimize(obj, z0, method='SLSQP', bounds=bnds,
                     constraints=list(cons)+list(extra),
                     options={'maxiter':800, 'ftol':ftol})
        if r.success:
            return r.x
    return None

def frontera(mu, S, w0, turnover, npts=35):
    cons, bnds, pick, z0 = _planteo(mu, w0, turnover)
    fvar = lambda z: pick(z) @ S @ pick(z)
    zmin = _resolver(fvar, cons, bnds, z0)
    zmax = _resolver(lambda z: -(mu @ pick(z)), cons, bnds, z0)
    out = []
    for th in np.linspace(mu @ pick(zmin), mu @ pick(zmax), npts):
        z = _resolver(fvar, cons, bnds, z0,
                      [{'type':'ineq','fun': (lambda z, th=th: mu @ pick(z) - th)}])
        if z is not None:
            w = pick(z)
            out.append((float(mu @ w), float(np.sqrt(w @ S @ w))))
    return pd.DataFrame(out, columns=['retorno','riesgo'])

def max_retorno(mu, S, s_obj, w0, turnover):
    """Maximo margen alcanzable sin superar un nivel de riesgo dado."""
    cons, bnds, pick, z0 = _planteo(mu, w0, turnover)
    z = _resolver(lambda z: -(mu @ pick(z)), cons, bnds, z0,
                  [{'type':'ineq','fun': lambda z: s_obj**2 - pick(z) @ S @ pick(z)}])
    return pick(z)

In [ ]:
r0 = float(mu_a @ w0)
s0 = float(np.sqrt(w0 @ S_a @ w0))

w_opt = max_retorno(mu_a, S_a, s0, w0, TURNOVER)
w_lib = max_retorno(mu_a, S_a, s0, w0, None)        # sin limite de rotacion
w_hist = max_retorno(mu_hist, S_a, s0, w0, TURNOVER) # robustez
FR = frontera(mu_a, S_a, w0, TURNOVER)

r1 = float(mu_a @ w_opt); s1 = float(np.sqrt(w_opt @ S_a @ w_opt))
rot = float(np.abs(w_opt - w0).sum() / 2)

print(f'Cartera vigente:   margen {r0:6.3f}  riesgo {s0:5.3f}  USD/m²/año')
print(f'Cartera propuesta: margen {r1:6.3f}  riesgo {s1:5.3f}  USD/m²/año')
print(f'\nCOSTO DE OPORTUNIDAD: {r1-r0:.3f} USD/m²/año  =  '
      f'{(r1-r0)*AREA_TOTAL:,.0f} USD/año')
print(f'Rotación aplicada: {rot*100:.1f} %  ({rot*AREA_TOTAL:,.0f} m²)')
print(f'Variedades eliminadas: {((w0>1e-4) & (w_opt<=1e-4)).sum()}')
print(f'\nRobustez — correlación de pesos con media histórica: '
      f'{np.corrcoef(w_opt, w_hist)[0,1]:.4f}')

### Figura 9. Frontera de eficiencia y costo de oportunidad

In [ ]:
sd_var = np.sqrt(np.diag(S_a))
fig, ax = plt.subplots(figsize=(7.5, 5.4))
ax.scatter(sd_var, mu_a, s=16, c=GR, edgecolor='#999999', lw=.3,
           label='Variedades individuales', zorder=1)
ax.plot(FR.riesgo, FR.retorno, '-o', ms=3.5, color=AZ, lw=1.8,
        label='Frontera de eficiencia', zorder=3)
ax.scatter([s0], [r0], s=150, marker='D', color=RO, zorder=5,
           edgecolor='k', lw=.6, label='Cartera vigente')
ax.scatter([s1], [r1], s=230, marker='*', color=VE, zorder=5,
           edgecolor='k', lw=.6, label='Cartera propuesta')
ax.annotate('', xy=(s1, r1), xytext=(s0, r0),
            arrowprops=dict(arrowstyle='->', color='k', lw=1.4))
ax.text(s0*1.02, (r0+r1)/2, f'+{r1-r0:.2f} USD/m²/año', fontsize=10)
ax.axhline(0, color='k', lw=.7)
ax.set_xlabel('Riesgo: desviación estándar del margen anual (USD por m²)')
ax.set_ylabel('Margen esperado (USD por m² al año)')
ax.set_title('Frontera de eficiencia y costo de oportunidad')
ax.legend(fontsize=9, loc='lower right')
guardar(fig, 'm09_frontera')

In [ ]:
A = pd.DataFrame({'variety': VAR,
                  'area_actual_m2': w0 * AREA_TOTAL,
                  'share_actual': w0 * 100,
                  'area_optima_m2': w_opt * AREA_TOTAL,
                  'share_optimo': w_opt * 100,
                  'margen_esperado_m2_anio': mu_a,
                  'sd_residual_anual': sd_var})
A['delta_m2'] = A.area_optima_m2 - A.area_actual_m2
A['delta_pp'] = A.share_optimo - A.share_actual
A['decision'] = np.select([A.delta_m2 > 50, A.delta_m2 < -50], ['Ampliar','Reducir'], 'Mantener')
A.sort_values('delta_m2', ascending=False).to_csv(SALIDA + 'asignacion.csv', index=False)
A.sort_values('delta_m2', ascending=False).head(10).round(2)

### Figura 10. Reasignación propuesta

In [ ]:
t = A.assign(abs_=A.delta_m2.abs()).nlargest(16, 'abs_').sort_values('delta_m2')
fig, ax = plt.subplots(figsize=(7.5, 5.6))
ax.barh(t.variety, t.delta_m2, color=np.where(t.delta_m2 > 0, VE, RO))
ax.axvline(0, color='k', lw=.9)
ax.set_xlabel('Cambio de superficie propuesto (m²)')
ax.set_title('Dieciséis movimientos de mayor magnitud')
guardar(fig, 'm10_reasignacion')

### Figura 11. Composición antes y después

In [ ]:
t = A.nlargest(12, 'share_actual').sort_values('share_actual')
y = np.arange(len(t))
fig, ax = plt.subplots(figsize=(7.5, 5.4))
ax.barh(y - .2, t.share_actual, height=.4, color=RO, label='Vigente')
ax.barh(y + .2, t.share_optimo, height=.4, color=VE, label='Propuesta')
ax.set_yticks(y); ax.set_yticklabels(t.variety)
ax.set_xlabel('Participación en la superficie (%)')
ax.set_title('Composición de la cartera antes y después de la optimización')
ax.legend()
guardar(fig, 'm11_composicion')

### Figura 12. La decisión no se reduce a un ordenamiento por margen

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(A.margen_esperado_m2_anio, A.delta_m2, s=A.area_actual_m2/45,
           alpha=.6, c=np.where(A.delta_m2 > 0, VE, RO), edgecolor='k', lw=.4)
ax.axhline(0, color='k', lw=.9); ax.axvline(0, color='k', lw=.9)
ax.set_xlabel('Margen esperado (USD por m² al año)')
ax.set_ylabel('Cambio de superficie propuesto (m²)')
ax.set_title('La decisión no se reduce a un ordenamiento por margen')
guardar(fig, 'm12_margen_vs_cambio')

# Casos que contradicen una regla de ordenamiento simple
print('Variedades de margen POSITIVO que el modelo elimina:')
print(A[(A.margen_esperado_m2_anio > 0) & (A.share_optimo < 1e-4)]
      [['variety','margen_esperado_m2_anio','sd_residual_anual','share_actual']]
      .round(2).to_string(index=False))

### Figura 13. Participación vigente frente a la propuesta

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.4))
ax.scatter(A.share_actual, A.share_optimo, s=26, alpha=.7, color=AZ, edgecolor='k', lw=.3)
lim = [0, max(A.share_actual.max(), A.share_optimo.max())*1.05]
ax.plot(lim, lim, 'k--', lw=1.2, label='Sin cambio')
ax.set_xlabel('Participación vigente (%)'); ax.set_ylabel('Participación propuesta (%)')
ax.set_title('Participación vigente frente a la propuesta por variedad')
ax.legend()
guardar(fig, 'm13_vigente_vs_propuesta')

## 5. Evaluación y resumen

### Métricas empleadas

| Etapa | Métrica | Por qué |
|---|---|---|
| Predictiva | RMSE | Penaliza errores grandes, que son los que distorsionan la matriz de riesgo. Expresada en las unidades de la variable. |
| Predictiva | MAE | Menos sensible a valores extremos; complementa al RMSE. |
| Predictiva | R² fuera de muestra | Proporción de varianza explicada frente a la media. |
| Predictiva | Mejora sobre la media histórica | La línea base **es** el estimador de la literatura clásica; superarla es la condición para justificar el modelo. |
| Predictiva | Autocorrelación de los residuos | Verifica el supuesto que sostiene la anualización del riesgo. |
| Prescriptiva | Número de condición y δ | Diagnostican la estabilidad numérica de la matriz de covarianzas. |
| Prescriptiva | Costo de oportunidad | Métrica de negocio: margen adicional a igual riesgo. |
| Prescriptiva | Correlación de pesos entre estimadores | Robustez de la recomendación. |

In [ ]:
resumen = {
 'observaciones_modelables': int(len(dm)),
 'mejor_modelo': MEJOR,
 'rmse': round(float(T.loc[MEJOR,'RMSE']), 4),
 'rmse_linea_base': round(float(base), 4),
 'mejora_rmse_pct': round(float(T.loc[MEJOR,'Mejora RMSE %']), 1),
 'r2': round(float(T.loc[MEJOR,'R2']), 4),
 'r2_linea_base': round(float(T.loc['Media histórica (línea base)','R2']), 4),
 'autocorr_residuos_lag1': round(float(acs[0]), 4),
 'shrinkage': round(float(lw.shrinkage_), 4),
 'cond_muestral': float(ev_s.max()/max(abs(ev_s.min()),1e-18)),
 'cond_encogida': round(float(ev_l.max()/ev_l.min()), 1),
 'area_total_m2': round(AREA_TOTAL),
 'margen_vigente': round(r0, 3),
 'margen_propuesto': round(r1, 3),
 'costo_oportunidad_usd_m2': round(r1-r0, 3),
 'costo_oportunidad_usd_anio': round((r1-r0)*AREA_TOTAL),
 'sin_rotacion_usd_anio': round(float(mu_a @ w_lib - r0)*AREA_TOTAL),
 'rotacion_pct': round(rot*100, 1),
 'variedades_eliminadas': int(((w0>1e-4) & (w_opt<=1e-4)).sum()),
 'robustez_corr_pesos': round(float(np.corrcoef(w_opt, w_hist)[0,1]), 3),
}
json.dump(resumen, open(SALIDA + 'resumen_modelo.json','w'), indent=1)
FR.to_csv(SALIDA + 'frontera.csv', index=False)
print(json.dumps(resumen, indent=1, ensure_ascii=False))